In [ ]:
# Script that plots apical areas, NB-NC contact lengths and durations for transcribing and non-transcribing NCs shown in Figure 3

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load dataset
df_original = pd.read_csv('CombinedMatrixPreWindowFirstSignFrameFINAL0925.csv')

print("\n=== ORIGINAL DATASET ===")
print("Total n:", len(df_original))
print("Original n per signalling group:")
print(df_original['signalling'].value_counts())

# Load dataset
df = pd.read_csv('CombinedMatrixPreWindowFirstSignFrameFINAL0925.csv')

# Clean data
df = df[df['contact_duration'] >= 0]
df = df.dropna(subset=['area', 'contact_length', 'contact_duration', 'signalling'])
df = df[df['area'] > 0]  # Avoid division by zero

# Add signalling label
df['signalling_label'] = df['signalling'].map({0: 'Non-signalling', 1: 'Signalling'})

# Compute ratio
df['length_area_ratio'] = df['contact_length'] / df['area']

# Variables to plot
variables = ['area', 'length_area_ratio', 'contact_duration']

# Summary statistics (mean, SD, n)
summary_stats = (
    df.groupby('signalling_label')[variables]
      .agg(['mean', 'std', 'count'])
      .round(3)
)

summary_stats.to_excel('signalling_summary_stats.xlsx')

print("\nSummary statistics (mean ± SD, n):\n")
print(summary_stats)
# Save cleaned dataset used for all plots
df.to_csv('dataset_used_for_all_plots.csv', index=False)
# Plotting
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, var in zip(axes, variables):

    sns.boxplot(
        x='signalling_label', y=var, data=df,
        palette='Set2', ax=ax, showfliers=False
    )

    sns.stripplot(
        x='signalling_label', y=var, data=df,
        color=".25", size=3, jitter=True, ax=ax
    )

    # Extract group data
    group1 = df.loc[df['signalling_label'] == 'Signalling', var]
    group0 = df.loc[df['signalling_label'] == 'Non-signalling', var]

    # Compute stats
    mean1, sd1, n1 = group1.mean(), group1.std(), group1.count()
    mean0, sd0, n0 = group0.mean(), group0.std(), group0.count()

    # T-test
    t_stat, p_val = ttest_ind(group1, group0, equal_var=False)

    if p_val < 0.001:
        significance = "***"
    elif p_val < 0.01:
        significance = "**"
    elif p_val < 0.05:
        significance = "*"
    else:
        significance = "n.s."

    # Annotation positions
    ymax = df[var].max()
    y_offset = ymax * 0.1
    ypos = ymax + y_offset

    # p-value annotation
    ax.text(0.5, ypos, f"p = {p_val:.3e} ({significance})",
            ha='center', va='bottom', fontsize=10)

    # Bracket
    ax.plot([0, 1], [ypos - y_offset*0.2, ypos - y_offset*0.2], color="black")

    # Add mean ± SD and n for each group
    ax.text(0, ymax*0.9,
            f"{mean0:.2f} ± {sd0:.2f}\n(n = {n0})",
            ha='center', fontsize=9)

    ax.text(1, ymax*0.9,
            f"{mean1:.2f} ± {sd1:.2f}\n(n = {n1})",
            ha='center', fontsize=9)

    # Force y-axis to start at 0
    ax.set_ylim(0, ypos + y_offset*0.5)

    ax.set_title(f'{var.replace("_", " ").capitalize()} by Signalling Status')
    ax.set_xlabel('')
    ax.set_ylabel(var.replace("_", " ").capitalize())

sns.despine()

plt.tight_layout()
plt.savefig('signalling_vs_nonsignalling_boxplots_with_ratio_and_ttests.pdf')
plt.show

